# 12_12_索引の種類と_再現率_速度

コード：`ch12/blocks/12_12_索引の種類と_再現率_速度.ipynb`

本文のコードをコードセルで直接実行します。

モデル・データの取得、または長い計算を含みます。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2', 'transformers==5.22.1', 'tiktoken==0.14.0', 'scikit-learn>=1.5,<2', 'pydantic==2.14.2', 'outlines==1.3.3', 'lancedb==0.41.0', 'pyarrow==26.0.1', 'sentence-transformers==6.2.0']], check=True)


In [ ]:
# 12 章：近似最近傍（IVF_PQ）の再現率と速度。総当たりの正解と比べる（LanceDB）
import time, shutil, numpy as np, lancedb
from lancedb.index import IvfPq, IvfSq, IvfFlat

rng = np.random.default_rng(0)
n, d, n_q, k = 100_000, 384, 200, 10
centers = rng.normal(size=(200, d))                       # 話題のかたまりを 200 個作る
X = centers[rng.integers(0, 200, n)] + 0.5 * rng.normal(size=(n, d))
X /= np.linalg.norm(X, axis=1, keepdims=True)              # 正規化（内積＝コサイン）
Q = X[rng.choice(n, n_q, replace=False)] + 0.05 * rng.normal(size=(n_q, d))
Q /= np.linalg.norm(Q, axis=1, keepdims=True)
X, Q = X.astype("float32"), Q.astype("float32")

t0 = time.perf_counter()
truth = np.argsort(-(Q @ X.T), axis=1)[:, :k]             # 総当たりの正解
brute_ms = (time.perf_counter() - t0) / n_q * 1000

shutil.rmtree("ann_index", ignore_errors=True)
tbl = lancedb.connect("ann_index").create_table("v", [{"id": i, "vector": v} for i, v in enumerate(X)])
def build(config):
    t0 = time.perf_counter()
    tbl.create_index("vector", config=config, replace=True)
    return time.perf_counter() - t0

def run(nprobes, refine=None):
    hits, t0 = 0, time.perf_counter()
    for qi, q in enumerate(Q):
        s = tbl.search(q).distance_type("dot").limit(k).nprobes(nprobes)
        if refine: s = s.refine_factor(refine)
        ids = [r["id"] for r in s.to_list()]
        hits += len(set(ids) & set(truth[qi]))
    return hits / (n_q * k), (time.perf_counter() - t0) / n_q * 1000

print(f"{n:,} 件・{d} 次元。総当たり（NumPy）1 問 {brute_ms:.2f} ms")
configs = {
    "IVF_PQ（48 分割・1 件 48 バイト）": IvfPq(distance_type="dot", num_partitions=256, num_sub_vectors=48),
    "IVF_PQ（96 分割・1 件 96 バイト）": IvfPq(distance_type="dot", num_partitions=256, num_sub_vectors=96),
    "IVF_SQ（8bit・1 件 384 バイト）":   IvfSq(distance_type="dot", num_partitions=256),
    "IVF_FLAT（圧縮なし・1 件 1,536 バイト）": IvfFlat(distance_type="dot", num_partitions=256),
}
for name, cfg in configs.items():
    sec = build(cfg)
    print(f"{name}  索引の作成 {sec:.1f} 秒")
    for nprobes, refine in ((1, None), (8, None), (32, None), (32, 10)):
        rec, ms = run(nprobes, refine)
        print(f"   nprobes={nprobes:2d}{'' if not refine else f' refine={refine}':10s} recall@10 {rec:.3f}  1 問 {ms:5.2f} ms")
